**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# 벡터 DB 심층 비교 분석 및 실습

## 학습 목표

1. 벡터 DB의 핵심 개념과 전통 DB와의 차이를 이해한다
2. 주요 벡터 DB(ChromaDB, FAISS, Pinecone, Weaviate, Milvus, pgvector)의 특징과 장단점을 비교한다
3. ChromaDB와 FAISS를 직접 실습하여 벡터 검색 파이프라인을 구축한다
4. 용도별 벡터 DB 선택 기준을 수립한다

---

### 실습 환경
- **GPU**: 불필요 (CPU로 충분)
- **필수 패키지**: chromadb, faiss-cpu, sentence-transformers, psycopg2-binary

In [ ]:
# 필수 패키지 설치
!pip install -q chromadb faiss-cpu sentence-transformers psycopg2-binary numpy

In [ ]:
# 패키지 버전 확인
import importlib

packages = [
    "chromadb",
    "faiss",
    "sentence_transformers",
    "numpy",
]

print("패키지 버전 확인")
print("=" * 40)

for pkg_name in packages:
    try:
        pkg = importlib.import_module(pkg_name)
        version = getattr(pkg, "__version__", "installed")
        print(f"  [OK] {pkg_name}: {version}")
    except ImportError:
        print(f"  [X]  {pkg_name}: 설치 필요")

---

## 1️⃣ 벡터 DB란? 전통 DB vs 벡터 DB 차이

### 전통 데이터베이스 (RDBMS)
- **저장 방식**: 행(row)과 열(column)로 구조화된 데이터
- **검색 방식**: SQL 쿼리, 정확한 키워드 매칭 (WHERE, LIKE)
- **적합한 작업**: 정형 데이터 관리, 트랜잭션 처리

### 벡터 데이터베이스
- **저장 방식**: 고차원 벡터(임베딩)로 변환된 비정형 데이터
- **검색 방식**: 유사도 검색 (코사인 유사도, 유클리드 거리, 내적)
- **적합한 작업**: 시맨틱 검색, 추천 시스템, RAG, 이미지 검색

### 비교표

| 구분 | 전통 DB (RDBMS) | 벡터 DB |
|------|-----------------|----------|
| **데이터 형태** | 정형 데이터 (숫자, 문자열) | 고차원 벡터 (임베딩) |
| **검색 방식** | 정확 매칭 (=, LIKE) | 유사도 기반 (ANN) |
| **인덱싱** | B-Tree, Hash | HNSW, IVF, PQ |
| **결과** | 정확히 일치하는 행 | 가장 유사한 K개 |
| **쿼리 예시** | `SELECT * WHERE name='AI'` | `query(vector, top_k=5)` |
| **주요 사례** | 금융, ERP, CRM | RAG, 추천, 이미지 검색 |

### 벡터 검색의 핵심: ANN (Approximate Nearest Neighbor)

```
  정확 검색 (Brute Force)         근사 검색 (ANN)
  ┌─────────────────┐          ┌─────────────────┐
  │ 모든 벡터와 비교   │          │ 인덱스로 후보 축소  │
  │ O(N) 시간 복잡도  │          │ O(log N) 수준     │
  │ 100% 정확        │          │ 99%+ 정확 (충분)   │
  │ 대규모 시 느림     │          │ 대규모에서도 빠름   │
  └─────────────────┘          └─────────────────┘
```

---

## 2️⃣ 주요 벡터 DB 비교: ChromaDB, FAISS, Pinecone, Weaviate, Milvus, PostgreSQL+pgvector

### 주요 벡터 DB 종합 비교표

| 항목 | ChromaDB | FAISS | Pinecone | Weaviate | Milvus | pgvector |
|------|----------|-------|----------|----------|--------|----------|
| **개발사** | Chroma | Meta | Pinecone | Weaviate | Zilliz | PostgreSQL 확장 |
| **라이선스** | Apache 2.0 | MIT | 상용 (무료 티어) | BSD-3 | Apache 2.0 | PostgreSQL |
| **배포 방식** | 임베디드/서버 | 라이브러리 | 클라우드 관리형 | 자체 호스팅/클라우드 | 자체 호스팅/클라우드 | PostgreSQL 확장 |
| **언어** | Python | C++/Python | REST API | Go/Python | Go/Python | C/SQL |
| **인덱스** | HNSW | IVF, PQ, HNSW 등 | 독자 알고리즘 | HNSW | IVF, HNSW, DiskANN | IVFFlat, HNSW |
| **확장성** | 소~중규모 | 대규모 (GPU 지원) | 대규모 | 대규모 | 초대규모 (10억+) | 중규모 |
| **메타데이터 필터** | O | X (직접 구현) | O | O | O | O (SQL) |
| **영속성** | O | 수동 저장 | 클라우드 자동 | O | O | O (PostgreSQL) |
| **난이도** | 매우 쉬움 | 중간 | 쉬움 | 중간 | 중~상 | SQL 필요 |

### 각 벡터 DB의 특징

**ChromaDB**
- 장점: API가 매우 직관적, Python 네이티브, 메타데이터 필터링 내장, 빠른 프로토타이핑
- 단점: 대규모 프로덕션에는 부족, GPU 미지원
- 추천: 학습, 프로토타입, 소규모 RAG 프로젝트

**FAISS (Facebook AI Similarity Search)**
- 장점: 초고속 검색, GPU 가속 지원, 다양한 인덱스 알고리즘, 메모리 효율적
- 단점: 메타데이터 필터링 없음, DB 기능 부재 (라이브러리), CRUD 제한적
- 추천: 대규모 벡터 검색, 성능이 최우선인 경우

**Pinecone**
- 장점: 완전 관리형 서비스, 설정 불필요, 자동 스케일링, 높은 가용성
- 단점: 유료(대규모 시), 벤더 종속, 데이터가 외부 클라우드에 저장
- 추천: 빠른 프로덕션 배포, 인프라 관리 부담을 줄이고 싶은 경우

**Weaviate**
- 장점: GraphQL 지원, 내장 벡터화 모듈, 하이브리드 검색 (키워드+벡터)
- 단점: 리소스 사용량 높음, 학습 곡선
- 추천: 복잡한 스키마가 필요한 경우, 하이브리드 검색

**Milvus**
- 장점: 10억 벡터 이상 처리 가능, 분산 아키텍처, 다양한 인덱스
- 단점: 설치/운영 복잡, 리소스 요구량 높음
- 추천: 초대규모 벡터 검색, 엔터프라이즈 환경

**PostgreSQL + pgvector**
- 장점: 기존 PostgreSQL 인프라 활용, SQL로 벡터 검색, 정형+벡터 데이터 통합
- 단점: 전용 벡터 DB 대비 성능 제한, 대규모 시 속도 저하
- 추천: 이미 PostgreSQL을 사용 중인 경우, 정형 데이터와 벡터를 함께 관리

---

## 3️⃣ ChromaDB 실습: 문서 임베딩, 저장, 유사도 검색

ChromaDB를 사용하여 문서를 임베딩하고, 저장한 뒤, 유사도 기반으로 검색합니다.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 임베딩 모델 로딩
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
print(f"임베딩 모델 로딩: {EMBEDDING_MODEL}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"임베딩 차원: {embedding_model.get_sentence_embedding_dimension()}")

# 실습용 한국어 샘플 문서
documents = [
    "벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니다. 전통적인 데이터베이스와 달리 유사도 기반 검색을 지원합니다.",
    "FAISS는 Meta에서 개발한 벡터 유사도 검색 라이브러리입니다. GPU 가속을 지원하여 수십억 개의 벡터도 빠르게 검색할 수 있습니다.",
    "ChromaDB는 오픈소스 벡터 데이터베이스로, Python에서 간단한 API로 임베딩을 저장하고 검색할 수 있습니다.",
    "RAG는 Retrieval-Augmented Generation의 약자로, 외부 지식을 검색하여 LLM의 답변 품질을 향상시키는 기술입니다.",
    "트랜스포머 아키텍처의 셀프 어텐션 메커니즘은 입력 시퀀스 내 모든 토큰 간의 관계를 동시에 계산합니다.",
    "Pinecone은 완전 관리형 벡터 데이터베이스 서비스로, 인프라 관리 없이 벡터 검색을 구현할 수 있습니다.",
    "pgvector는 PostgreSQL의 확장으로, 기존 SQL 데이터베이스에 벡터 검색 기능을 추가할 수 있습니다.",
    "임베딩은 텍스트, 이미지 등의 비정형 데이터를 고차원 벡터로 변환하는 과정입니다. 의미가 유사한 데이터는 벡터 공간에서 가까이 위치합니다.",
    "HNSW 알고리즘은 계층적 탐색 가능한 소세계 그래프를 구축하여 근사 최근접 이웃 검색을 수행합니다.",
    "코사인 유사도는 두 벡터 간의 각도를 기반으로 유사도를 측정합니다. 값이 1에 가까울수록 유사하고 0에 가까울수록 다릅니다.",
]

metadatas = [
    {"category": "vector_db", "topic": "개념"},
    {"category": "vector_db", "topic": "FAISS"},
    {"category": "vector_db", "topic": "ChromaDB"},
    {"category": "rag", "topic": "RAG"},
    {"category": "model", "topic": "트랜스포머"},
    {"category": "vector_db", "topic": "Pinecone"},
    {"category": "vector_db", "topic": "pgvector"},
    {"category": "embedding", "topic": "임베딩"},
    {"category": "algorithm", "topic": "HNSW"},
    {"category": "algorithm", "topic": "코사인유사도"},
]

print(f"\n샘플 문서 {len(documents)}개 준비 완료")

In [ ]:
import chromadb

# ChromaDB 클라이언트 생성 (인메모리)
chroma_client = chromadb.Client()

# 컬렉션 생성
collection_name = "vector_db_comparison"
try:
    chroma_client.delete_collection(collection_name)
except:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)

# 임베딩 생성 및 저장
doc_embeddings = embedding_model.encode(documents).tolist()
ids = [f"doc_{i}" for i in range(len(documents))]

collection.add(
    ids=ids,
    embeddings=doc_embeddings,
    documents=documents,
    metadatas=metadatas,
)

print(f"ChromaDB 컬렉션 '{collection_name}' 생성 완료")
print(f"저장된 문서 수: {collection.count()}")

In [ ]:
# ChromaDB 유사도 검색 실습
test_queries = [
    "벡터 검색에 적합한 데이터베이스는?",
    "GPU를 활용한 빠른 벡터 검색",
    "텍스트를 벡터로 변환하는 방법",
]

print("=" * 70)
print("ChromaDB 유사도 검색 결과")
print("=" * 70)

for query in test_queries:
    query_embedding = embedding_model.encode([query]).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3,
        include=["documents", "metadatas", "distances"]
    )
    
    print(f"\n[쿼리] {query}")
    for i in range(len(results['documents'][0])):
        doc = results['documents'][0][i]
        meta = results['metadatas'][0][i]
        distance = results['distances'][0][i]
        similarity = 1 - distance
        print(f"  {i+1}. (유사도: {similarity:.4f}) [{meta['topic']}] {doc[:60]}...")
    print("-" * 70)

# 메타데이터 필터링 검색
print("\n[메타데이터 필터] category='vector_db'인 문서만 검색")
query_embedding = embedding_model.encode(["데이터베이스 추천"]).tolist()
filtered_results = collection.query(
    query_embeddings=query_embedding,
    n_results=3,
    where={"category": "vector_db"},
    include=["documents", "metadatas", "distances"]
)

for i in range(len(filtered_results['documents'][0])):
    doc = filtered_results['documents'][0][i]
    meta = filtered_results['metadatas'][0][i]
    similarity = 1 - filtered_results['distances'][0][i]
    print(f"  {i+1}. (유사도: {similarity:.4f}) [{meta['topic']}] {doc[:60]}...")

---

## 4️⃣ FAISS 실습: 인덱스 생성, 검색 성능 비교

FAISS는 Meta에서 개발한 고성능 벡터 검색 라이브러리입니다.
다양한 인덱스 유형을 제공하며, 대규모 데이터에서 빠른 검색이 가능합니다.

### FAISS 주요 인덱스 유형

| 인덱스 | 설명 | 속도 | 정확도 | 메모리 |
|--------|------|------|--------|--------|
| `IndexFlatL2` | 완전 탐색 (L2 거리) | 느림 | 100% | 높음 |
| `IndexFlatIP` | 완전 탐색 (내적) | 느림 | 100% | 높음 |
| `IndexIVFFlat` | 클러스터 기반 검색 | 빠름 | 높음 | 중간 |
| `IndexIVFPQ` | 양자화 압축 + 클러스터 | 매우 빠름 | 중간 | 낮음 |
| `IndexHNSW` | 그래프 기반 검색 | 매우 빠름 | 높음 | 높음 |

In [ ]:
import faiss
import numpy as np
import time

# 동일한 문서의 임베딩을 numpy 배열로 변환
doc_vectors = embedding_model.encode(documents)
doc_vectors = np.array(doc_vectors).astype('float32')
dimension = doc_vectors.shape[1]

print(f"벡터 차원: {dimension}")
print(f"벡터 개수: {doc_vectors.shape[0]}")

# --- IndexFlatL2 (완전 탐색, L2 거리) ---
print("\n" + "=" * 50)
print("[1] IndexFlatL2 - 완전 탐색 (Brute Force)")
print("=" * 50)

index_flat = faiss.IndexFlatL2(dimension)
index_flat.add(doc_vectors)
print(f"인덱스에 저장된 벡터 수: {index_flat.ntotal}")

# 검색
query_text = "벡터 검색에 적합한 데이터베이스는?"
query_vector = embedding_model.encode([query_text]).astype('float32')

start = time.time()
distances, indices = index_flat.search(query_vector, k=3)
elapsed = time.time() - start

print(f"\n[쿼리] {query_text}")
print(f"검색 시간: {elapsed*1000:.3f}ms")
for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"  {i+1}. (L2 거리: {dist:.4f}) {documents[idx][:60]}...")

In [ ]:
# 대규모 데이터에서 FAISS 인덱스 성능 비교
# 랜덤 벡터 10만 개 생성하여 인덱스별 속도 비교

np.random.seed(42)
n_vectors = 100_000
dim = 384  # MiniLM 임베딩 차원
large_vectors = np.random.random((n_vectors, dim)).astype('float32')
query_vectors = np.random.random((10, dim)).astype('float32')  # 10개 쿼리

print(f"테스트 벡터: {n_vectors:,}개 x {dim}차원")
print(f"쿼리 벡터: {query_vectors.shape[0]}개")
print()

# --- IndexFlatL2 ---
print("[1] IndexFlatL2 (완전 탐색)")
index_flat = faiss.IndexFlatL2(dim)
start = time.time()
index_flat.add(large_vectors)
build_time = time.time() - start

start = time.time()
distances, indices = index_flat.search(query_vectors, k=5)
search_time = time.time() - start
print(f"  인덱스 생성: {build_time*1000:.1f}ms | 검색(10쿼리): {search_time*1000:.1f}ms")

# --- IndexIVFFlat ---
print("\n[2] IndexIVFFlat (클러스터 기반)")
nlist = 100  # 클러스터 수
quantizer = faiss.IndexFlatL2(dim)
index_ivf = faiss.IndexIVFFlat(quantizer, dim, nlist)

start = time.time()
index_ivf.train(large_vectors)  # 학습 필요
index_ivf.add(large_vectors)
build_time = time.time() - start

index_ivf.nprobe = 10  # 검색할 클러스터 수
start = time.time()
distances_ivf, indices_ivf = index_ivf.search(query_vectors, k=5)
search_time = time.time() - start
print(f"  인덱스 생성: {build_time*1000:.1f}ms | 검색(10쿼리): {search_time*1000:.1f}ms")

# --- IndexHNSWFlat ---
print("\n[3] IndexHNSWFlat (그래프 기반)")
index_hnsw = faiss.IndexHNSWFlat(dim, 32)  # M=32 (연결 수)

start = time.time()
index_hnsw.add(large_vectors)
build_time = time.time() - start

start = time.time()
distances_hnsw, indices_hnsw = index_hnsw.search(query_vectors, k=5)
search_time = time.time() - start
print(f"  인덱스 생성: {build_time*1000:.1f}ms | 검색(10쿼리): {search_time*1000:.1f}ms")

# Recall 비교 (FlatL2를 정답으로 사용)
print("\n" + "=" * 50)
print("정확도 비교 (Recall@5, FlatL2 기준)")
print("=" * 50)

def compute_recall(gt_indices, test_indices, k=5):
    """정답 인덱스 대비 recall 계산"""
    recalls = []
    for gt, test in zip(gt_indices, test_indices):
        recall = len(set(gt[:k]) & set(test[:k])) / k
        recalls.append(recall)
    return np.mean(recalls)

recall_ivf = compute_recall(indices, indices_ivf)
recall_hnsw = compute_recall(indices, indices_hnsw)
print(f"  IndexIVFFlat  Recall@5: {recall_ivf:.2%}")
print(f"  IndexHNSWFlat Recall@5: {recall_hnsw:.2%}")

---

## 5️⃣ PostgreSQL + pgvector 소개 및 간단 실습

**pgvector**는 PostgreSQL의 확장으로, 기존 관계형 DB에 벡터 검색 기능을 추가합니다.

### pgvector 장점
- 기존 PostgreSQL 인프라를 그대로 활용
- SQL과 벡터 검색을 하나의 쿼리로 결합 가능
- ACID 트랜잭션 보장
- 정형 데이터 + 벡터 데이터 통합 관리

### pgvector 설치 (참고)
```sql
-- PostgreSQL에서 pgvector 확장 설치
CREATE EXTENSION vector;
```

### pgvector 사용 예시 (SQL)
```sql
-- 벡터 컬럼이 있는 테이블 생성
CREATE TABLE documents (
    id SERIAL PRIMARY KEY,
    content TEXT,
    category VARCHAR(50),
    embedding vector(384)  -- 384차원 벡터
);

-- 벡터 인덱스 생성 (HNSW)
CREATE INDEX ON documents 
USING hnsw (embedding vector_cosine_ops);

-- 벡터 삽입
INSERT INTO documents (content, category, embedding) 
VALUES ('벡터 DB 소개', 'tech', '[0.1, 0.2, ...]');

-- 유사도 검색 (코사인 거리)
SELECT content, category,
       1 - (embedding <=> '[0.1, 0.2, ...]') AS similarity
FROM documents
WHERE category = 'tech'  -- SQL 필터와 벡터 검색 결합!
ORDER BY embedding <=> '[0.1, 0.2, ...]'
LIMIT 5;
```

### pgvector 거리 연산자
| 연산자 | 거리 유형 | 설명 |
|--------|-----------|------|
| `<->` | L2 거리 | 유클리드 거리 |
| `<=>` | 코사인 거리 | 1 - 코사인 유사도 |
| `<#>` | 내적 거리 | 음의 내적 |

In [ ]:
# pgvector 시뮬레이션 (PostgreSQL 없이 개념 실습)
# 실제 환경에서는 psycopg2로 PostgreSQL에 연결하여 사용합니다.

print("pgvector 개념 시뮬레이션 (Python으로 구현)")
print("=" * 60)

# SQL-like 구조를 Python dict로 시뮬레이션
class PgVectorSimulator:
    """pgvector의 동작을 시뮬레이션하는 클래스"""
    
    def __init__(self, dimension):
        self.dimension = dimension
        self.table = []  # [{id, content, category, embedding}, ...]
    
    def insert(self, content, category, embedding):
        """INSERT INTO documents ..."""
        self.table.append({
            "id": len(self.table) + 1,
            "content": content,
            "category": category,
            "embedding": np.array(embedding)
        })
    
    def query(self, query_embedding, where_category=None, limit=3):
        """
        SELECT content, category, 1-(embedding <=> query) AS similarity
        FROM documents
        WHERE category = ... 
        ORDER BY embedding <=> query
        LIMIT ...
        """
        query_vec = np.array(query_embedding)
        results = []
        
        for row in self.table:
            # WHERE 필터
            if where_category and row["category"] != where_category:
                continue
            
            # 코사인 유사도 계산 (<=>) 
            cos_sim = np.dot(row["embedding"], query_vec) / (
                np.linalg.norm(row["embedding"]) * np.linalg.norm(query_vec)
            )
            results.append({**row, "similarity": float(cos_sim)})
        
        # ORDER BY similarity DESC
        results.sort(key=lambda x: x["similarity"], reverse=True)
        return results[:limit]

# 시뮬레이터 생성 및 데이터 삽입
pgvec = PgVectorSimulator(dimension=dimension)

for doc, meta, emb in zip(documents, metadatas, doc_vectors):
    pgvec.insert(content=doc, category=meta["category"], embedding=emb)

print(f"테이블에 {len(pgvec.table)}개 행 삽입 완료\n")

# SQL 스타일 쿼리
query_text = "벡터 검색에 적합한 데이터베이스는?"
query_vec = embedding_model.encode([query_text])[0]

print(f"-- SELECT * WHERE category='vector_db' ORDER BY similarity LIMIT 3")
print(f"-- 쿼리: '{query_text}'\n")

results = pgvec.query(query_vec, where_category="vector_db", limit=3)
for r in results:
    print(f"  id={r['id']} | similarity={r['similarity']:.4f} | {r['category']}")
    print(f"    {r['content'][:70]}...")
    print()

print("-- 필터 없이 전체 검색")
results_all = pgvec.query(query_vec, limit=3)
for r in results_all:
    print(f"  id={r['id']} | similarity={r['similarity']:.4f} | {r['category']}")
    print(f"    {r['content'][:70]}...")
    print()

---

## 6️⃣ 벡터 DB 선택 가이드: 용도별 추천

### 의사결정 플로우

```
프로젝트 시작
  │
  ├─ 학습/프로토타입 목적? ──────────── YES ──> ChromaDB
  │
  ├─ 이미 PostgreSQL 사용 중? ────────── YES ──> pgvector
  │
  ├─ 인프라 관리 없이 빠르게 배포? ───── YES ──> Pinecone
  │
  ├─ 최고 검색 속도 필요? (GPU 활용) ── YES ──> FAISS
  │
  ├─ 하이브리드 검색 필요? ──────────── YES ──> Weaviate
  │
  └─ 10억+ 벡터 대규모 시스템? ──────── YES ──> Milvus
```

### 시나리오별 추천

| 시나리오 | 추천 DB | 이유 |
|----------|---------|------|
| 개인 RAG 챗봇 | **ChromaDB** | 간단한 설치, 빠른 프로토타이핑 |
| 사내 문서 검색 시스템 | **Weaviate** / **pgvector** | 하이브리드 검색, 기존 인프라 활용 |
| 대규모 이커머스 추천 | **Milvus** / **FAISS** | 수억 상품 벡터, 고속 검색 |
| 스타트업 MVP | **Pinecone** | 관리 부담 최소, 빠른 출시 |
| 연구/실험 | **FAISS** | 다양한 인덱스 실험, GPU 가속 |
| 기존 PostgreSQL 앱 확장 | **pgvector** | 마이그레이션 불필요, SQL 통합 |

### 비용 관점 비교

| 구분 | 무료 | 조건부 무료 | 유료 |
|------|------|------------|------|
| **완전 무료** | ChromaDB, FAISS, pgvector | - | - |
| **프리 티어** | - | Pinecone (제한적), Weaviate Cloud | - |
| **자체 호스팅 무료** | - | Milvus, Weaviate | 운영 비용 |
| **관리형 유료** | - | - | Pinecone Pro, Zilliz Cloud |

In [ ]:
# ChromaDB vs FAISS 직접 비교 실험
import time

print("=" * 60)
print("ChromaDB vs FAISS 직접 비교")
print("=" * 60)

# 동일한 쿼리로 두 DB에서 검색
comparison_queries = [
    "GPU를 사용한 대규모 벡터 검색",
    "간단하게 사용할 수 있는 벡터 DB",
    "코사인 유사도란 무엇인가",
]

# FAISS 인덱스 (이전에 만든 것 재사용)
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(doc_vectors)

for query in comparison_queries:
    print(f"\n[쿼리] {query}")
    query_vec = embedding_model.encode([query])
    query_vec_np = np.array(query_vec).astype('float32')
    
    # ChromaDB 검색
    start = time.time()
    chroma_results = collection.query(
        query_embeddings=query_vec.tolist(),
        n_results=2,
        include=["documents", "distances"]
    )
    chroma_time = (time.time() - start) * 1000
    
    # FAISS 검색
    start = time.time()
    faiss_dist, faiss_idx = faiss_index.search(query_vec_np, k=2)
    faiss_time = (time.time() - start) * 1000
    
    print(f"  ChromaDB ({chroma_time:.2f}ms):")
    for i, doc in enumerate(chroma_results['documents'][0]):
        sim = 1 - chroma_results['distances'][0][i]
        print(f"    {i+1}. (유사도: {sim:.4f}) {doc[:50]}...")
    
    print(f"  FAISS    ({faiss_time:.2f}ms):")
    for i, idx in enumerate(faiss_idx[0]):
        print(f"    {i+1}. (L2거리: {faiss_dist[0][i]:.4f}) {documents[idx][:50]}...")
    
    print("-" * 60)

In [ ]:
# 전체 실습 요약
print("=" * 60)
print("벡터 DB 심층 비교 분석 - 핵심 정리")
print("=" * 60)
print()
print("1. 벡터 DB는 고차원 벡터의 유사도 검색에 최적화된 데이터베이스")
print("   - 전통 DB: 정확 매칭 | 벡터 DB: 의미 기반 유사도 검색")
print()
print("2. 주요 벡터 DB 특징")
print("   - ChromaDB: 쉬운 API, 프로토타이핑에 최적")
print("   - FAISS: 최고 성능, GPU 가속, 다양한 인덱스")
print("   - Pinecone: 완전 관리형, 빠른 배포")
print("   - Weaviate: 하이브리드 검색, GraphQL")
print("   - Milvus: 초대규모(10억+), 분산 시스템")
print("   - pgvector: PostgreSQL 통합, SQL 활용")
print()
print("3. 인덱스 알고리즘별 트레이드오프")
print("   - FlatL2: 정확하지만 느림 (소규모 적합)")
print("   - IVFFlat: 빠르지만 학습 필요 (중규모 적합)")
print("   - HNSW: 빠르고 정확하지만 메모리 많이 사용")
print()
print("4. 선택 기준")
print("   - 데이터 규모, 검색 속도 요구, 인프라 환경,")
print("     메타데이터 필터링 필요 여부에 따라 결정")

---

## 실습 과제

1. ChromaDB에 자신만의 문서 5개를 추가하고, 다양한 쿼리로 검색 결과를 확인해보세요
2. FAISS에서 `IndexIVFFlat`의 `nprobe` 값을 1, 5, 10, 50으로 변경하며 recall과 속도의 변화를 관찰하세요
3. 동일한 쿼리에 대해 ChromaDB와 FAISS의 검색 결과 순위가 어떻게 다른지 비교해보세요
4. (심화) pgvector를 실제 PostgreSQL에 설치하고, SQL로 벡터 검색을 구현해보세요

---

## 참고 자료

- [ChromaDB 공식 문서](https://docs.trychroma.com/)
- [FAISS GitHub](https://github.com/facebookresearch/faiss)
- [FAISS 위키](https://github.com/facebookresearch/faiss/wiki)
- [pgvector GitHub](https://github.com/pgvector/pgvector)
- [Pinecone 문서](https://docs.pinecone.io/)
- [Weaviate 문서](https://weaviate.io/developers/weaviate)
- [Milvus 문서](https://milvus.io/docs)